# Miller Amplifier Example

## Python Setup

In [5]:
from pygmid import Lookup as lk
import numpy as np
from scipy.constants import Boltzmann, k
from IPython.display import HTML, display

nmos = lk('../mat/nfet_01v8_lvt.mat')
pmos = lk('../mat/pfet_01v8_lvt.mat')

## Input Parameters

In [6]:
# Circuit Parameters
vdd = 1.8 # V
ugf = 50e6 # Hz
cload = 1e-12 # F
noise = 10e-9 # V/sqrt(Hz)

# Device Parameters
## gm/ID
gm_id_M12 = 20
gm_id_M34 = 6
gm_id_M5 = 10
gm_id_M6 = gm_id_M34
gm_id_M7 = gm_id_M5

## Lengths
l_M12 = 0.5
l_M34 = 1
l_M5 = 1
l_M6 = l_M34
l_M7 = l_M5

## Calculations

In [7]:
# Overdrive Voltages (Vdsat [V])
vdsat_M12 = 2 / gm_id_M12
vdsat_M34 = 2 / gm_id_M34
vdsat_M5 = 2 / gm_id_M5
vdsat_M6 = 2 / gm_id_M6
vdsat_M7 = 2 / gm_id_M7

# Gate-Source Voltages (Vgs [V])
vgs_M12 = nmos.lookupVGS(GM_ID=gm_id_M12, VDS=vdsat_M12, VSB=vdsat_M5, L=l_M12)
vgs_M34 = pmos.lookupVGS(GM_ID=gm_id_M34, VSB=0, L=l_M34)
vgs_M34 = pmos.lookupVGS(GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34)
vgs_M5 = nmos.lookupVGS(GM_ID=gm_id_M5, VDS=vdsat_M5, VSB=0, L=l_M5)
vgs_M6 = pmos.lookupVGS(GM_ID=gm_id_M6, VDS=vdsat_M6, VSB=0, L=l_M6)
vgs_M7 = nmos.lookupVGS(GM_ID=gm_id_M7, VDS=vdsat_M7, VSB=0, L=l_M7)

# M1/2 γ(gamma), Transconductance, Current, Current Density, Width, Drain Capacitance
gamma_M12 = nmos.gamma(GM_ID=gm_id_M12, VDS=vdsat_M12, VSB=vdsat_M5, L=l_M12)
gm_M12 = 4 * k * 300 * gamma_M12 / noise**2
id_M12 = gm_M12 / gm_id_M12
jd_M12 = nmos.lookup('id_W', GM_ID=gm_id_M12, VDS=vdsat_M12, VSB=vdsat_M5, L=l_M12)
w_M12 = id_M12 / jd_M12
cdd_M12 = gm_M12 / nmos.lookup('GM_CDD', GM_ID=gm_id_M12, VDS=vdsat_M12, VSB=vdsat_M5, L=l_M12)

# M3/4 Current, Transconductance, Current Density, Width, Drain Capacitance
id_M34 = id_M12
gm_M34 = gm_id_M34 * id_M34
jd_M34 = pmos.lookup('id_W', GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34)
w_M34 = id_M34 / jd_M34
cdd_M34 = pmos.lookup('CDD_W', GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34) * w_M34

# 'Optimize' Loop
cself_IN = cself_OUT = 0
cself_IN = cdd_M12 + cdd_M34
Cc = gm_M12 / (2 * np.pi * ugf) - cself_IN
for i in range(20):
    gm_M6 = gm_M12 * 2 * (cload + cself_OUT) / (Cc + cself_IN)
    id_M6 = gm_M6 / gm_id_M6
    cdd_M6 = gm_M6 / pmos.lookup('GM_CDD', GM_ID=gm_id_M6, VDS=vdd/2, VSB=0, L=l_M6)

    id_M7 = id_M6
    gm_M7 = gm_id_M7 * id_M7
    cdd_M7 = gm_M7 / nmos.lookup('GM_CDD', GM_ID=gm_id_M7, VDS=vdd/2, VSB=0, L=l_M7)

    cself_OUT = cdd_M6 + cdd_M7

# M6Current Density, Width
jd_M6 = pmos.lookup('id_W', GM_ID=gm_id_M6, VDS=vdd/2, VSB=0, L=l_M6)
w_M6 = id_M6 / jd_M6

# M5/7 Current, Transconductance, Current Density, Width
id_M5 = id_M12 * 2
gm_M5 = gm_id_M5 * id_M5
jd_M5 = nmos.lookup('id_W', GM_ID=gm_id_M5, VDS=vdsat_M5, VSB=0, L=l_M5)
w_M5 = id_M5 / jd_M5
jd_M7 = jd_M5
w_M7 = w_M5 * id_M7 / id_M5

# Intrinsic Gains (gm/gds [V/V])
av_M12 = nmos.lookup('GM_GDS', GM_ID=gm_id_M12, VDS=vdd-vgs_M34-vdsat_M5, VSB=vdsat_M5, L=l_M12)
av_M34 = pmos.lookup('GM_GDS', GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34)
av_M6 = pmos.lookup('GM_GDS', GM_ID=gm_id_M6, VDS=vdd/2, VSB=0, L=l_M6)
av_M7 = nmos.lookup('GM_GDS', GM_ID=gm_id_M7, VDS=vdd/2, VSB=0, L=l_M7)
av_M5 = nmos.lookup('GM_GDS', GM_ID=gm_id_M5, VDS=vdsat_M5, VSB=0, L=l_M5)

# Output Conductances (gds [S])
gds_M12 = gm_M12 / av_M12
gds_M34 = gm_M34 / av_M34
gds_M6 = gm_M6 / av_M6
gds_M7 = gm_M7 / av_M7
gds_M5 = gm_M5 / av_M5

# Threshold Voltages (Vth [V])
vth_M12 = nmos.lookup('VT', vgs=vgs_M12, VDS=vdsat_M12, VSB=vdsat_M5, L=l_M12)
vth_M34 = pmos.lookup('VT', vgs=vgs_M34, VDS=vgs_M34, VSB=0, L=l_M34)
vth_M5 = nmos.lookup('VT', vgs=vgs_M5, VDS=vdsat_M5, VSB=0, L=l_M5)
vth_M6 = pmos.lookup('VT', vgs=vgs_M6, VDS=vdsat_M6, VSB=0, L=l_M6)
vth_M7 = nmos.lookup('VT', vgs=vgs_M7, VDS=vdsat_M7, VSB=0, L=l_M7)

# Circuit Gain (A0 [V/V, dB])
av1 = gm_M12 / (gds_M12 + gds_M34)
av2 = gm_M6 / (gds_M6 + gds_M7)
a0_Mag = av1 * av2
a0_dB = 20 * np.log10(a0_Mag)

# Circuit Consumption (IDD [A])
idd = id_M5 + id_M7

## Output

In [8]:
html_table = f"""
<style>
 .param-title {{ font-size: 18px; font-weight: bold; margin: 20px 0 10px 0; }}
 .param-table {{ border-collapse: collapse; margin: 20px 0; font-size: 14px; }}
 .param-table th, .param-table td {{ border: 1px solid #ddd; padding: 8px 12px; text-align: right; }}
 .param-table th {{ background-color: #000000; color: white; font-weight: bold; }}
 .param-table td:first-child {{ text-align: left; font-weight: 500; }}
 .param-table td:last-child {{ text-align: center; }}
 .circuit-params {{ font-family: monospace; margin: 20px 0; line-height: 1.6; }}
</style>
<div class="param-title">MOSFET PARAMETERS:</div>
<table class="param-table">
 <tr>
 <th>Parameter</th><th>M1/2</th><th>M3/4</th><th>M5</th><th>M6</th><th>M7</th><th>Unit</th>
 </tr>
 <tr><td>W</td><td>{w_M12:.3f}</td><td>{w_M34:.3f}</td><td>{w_M5:.3f}</td><td>{w_M6:.3f}</td><td>{w_M7:.3f}</td><td>µm</td></tr>
 <tr><td>L</td><td>{l_M12:.3f}</td><td>{l_M34:.3f}</td><td>{l_M5:.3f}</td><td>{l_M6:.3f}</td><td>{l_M7:.3f}</td><td>µm</td></tr>
 <tr><td>gm/ID</td><td>{gm_id_M12:.3f}</td><td>{gm_id_M34:.3f}</td><td>{gm_id_M5:.3f}</td><td>{gm_id_M6:.3f}</td><td>{gm_id_M7:.3f}</td><td>S/A</td></tr>
 <tr><td>JD</td><td>{jd_M12/1e-6:.3f}</td><td>{jd_M34/1e-6:.3f}</td><td>{jd_M5/1e-6:.3f}</td><td>{jd_M6/1e-6:.3f}</td><td>{jd_M7/1e-6:.3f}</td><td>µA/µm</td></tr>
 <tr><td>gm/gds</td><td>{av_M12:.3f}</td><td>{av_M34:.3f}</td><td>{av_M5:.3f}</td><td>{av_M6:.3f}</td><td>{av_M7:.3f}</td><td>V/V</td></tr>
 <tr><td>gm</td><td>{gm_M12/1e-6:.3f}</td><td>{gm_M34/1e-6:.3f}</td><td>{gm_M5/1e-6:.3f}</td><td>{gm_M6/1e-6:.3f}</td><td>{gm_M7/1e-6:.3f}</td><td>µS</td></tr>
 <tr><td>ID</td><td>{id_M12/1e-6:.3f}</td><td>{id_M34/1e-6:.3f}</td><td>{id_M5/1e-6:.3f}</td><td>{id_M6/1e-6:.3f}</td><td>{id_M7/1e-6:.3f}</td><td>µA</td></tr>
 <tr><td>gds</td><td>{gds_M12/1e-6:.3f}</td><td>{gds_M34/1e-6:.3f}</td><td>{gds_M5/1e-6:.3f}</td><td>{gds_M6/1e-6:.3f}</td><td>{gds_M7/1e-6:.3f}</td><td>µS</td></tr>
 <tr><td>Vdsat</td><td>{vdsat_M12/1e-3:.3f}</td><td>{vdsat_M34/1e-3:.3f}</td><td>{vdsat_M5/1e-3:.3f}</td><td>{vdsat_M6/1e-3:.3f}</td><td>{vdsat_M7/1e-3:.3f}</td><td>mV</td></tr>
 <tr><td>Vth</td><td>{vth_M12/1e-3:.3f}</td><td>{vth_M34/1e-3:.3f}</td><td>{vth_M5/1e-3:.3f}</td><td>{vth_M6/1e-3:.3f}</td><td>{vth_M7/1e-3:.3f}</td><td>mV</td></tr>
</table>
<div class="param-title">CIRCUIT PARAMETERS:</div>
<div class="circuit-params">
A0 = {a0_dB:.3f} dB, {a0_Mag:.3f} V/V<br>
UGF ≈ {ugf/1e6:.3f} MHz<br>
IDD = {idd/1e-6:.3f} µA<br>
Cc = {Cc/1e-15:.3f} fF<br>
CL = {cload/1e-12:.3f} pF
</div>
"""
display(HTML(html_table))

Parameter,M1/2,M3/4,M5,M6,M7,Unit
W,14.770,3.901,5.976,35.225,27.095,µm
L,0.500,1.000,1.000,1.000,1.000,µm
gm/ID,20.000,6.000,10.000,6.000,10.000,S/A
JD,0.805,3.048,3.980,3.061,3.980,µA/µm
gm/gds,87.473,55.809,12.860,60.855,88.556,V/V
gm,237.838,71.351,237.838,646.989,1078.315,µS
ID,11.892,11.892,23.784,107.831,107.831,µA
gds,2.719,1.278,18.494,10.632,12.177,µS
Vdsat,100.000,333.333,200.000,333.333,200.000,mV
Vth,554.200,444.260,483.100,444.200,483.100,mV
